# Lab 07 — Data Leakage & Validation Design

*AI Cybersecurity Masterclass · Lab 07 · AI60secLab*

**Companion video:** [▶ Is Your Cybersecurity AI Model Cheating?](https://www.youtube.com/watch?v=I-5g9lnQ9t4&list=PLG4l6Ssi9A00)

## Research question
Is the model learning real attack patterns, or an accidental shortcut?

Labs 01–06 produced scores. This lab asks whether those scores are trustworthy. Leakage is when the model sees information it would not have in production — the score goes up, and the model gets worse in the real world.

## Learning objectives
- confirm the answer is not hiding inside the feature matrix;
- fit preprocessing on training data only, and see what happens when you don't;
- wrap preprocessing and the model in one `Pipeline` so leakage is hard to cause by accident;
- check for duplicate rows shared across train and test;
- run a shuffled-label sanity test that a real result must fail;
- document the validation design so a reader can judge whether to believe you.

## Security foundation: trust, and how tests lie

Security runs on evidence. When someone claims a control works, the question is always *how do you know?* A test that leaks information is evidence that cannot be trusted, and decisions built on it fail in production.

Leakage has real-world equivalents in security testing:
- testing a detector on attacks it was built from (like grading with the answer key);
- a penetration test where the defenders were told in advance when it would happen;
- a vendor demo on the vendor's own curated traffic.

The habit this lab builds, separating what a model *could have known* from what it *would know in production*, is the same habit good security assessors use when they judge any claim.

In [1]:
# Setup: works in Google Colab and locally. No uploads needed.
# In Colab this clones the course repo, which already includes the UNSW-NB15 data.
from pathlib import Path
import sys, os

REPO_URL = "https://github.com/Jacquelinepersha/ai-cybersecurity-public-labs.git"
REPO_NAME = "ai-cybersecurity-public-labs"

try:
    import google.colab  # noqa: F401
    if not Path(REPO_NAME).exists():
        !git clone --depth 1 {REPO_URL}
    os.chdir(REPO_NAME)
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data" / "raw"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data folder :", DATA_DIR)

Project root: /home/claude/run/l07
Data folder : /home/claude/run/l07/data/raw


In [2]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

from src.data_loader import load_unsw
from src.preprocessing import split_xy, align_columns, build_preprocessor
from src.features import drop_identifier_like_columns
from src.evaluation import binary_metrics, binary_metrics_frame

train, test = load_unsw(DATA_DIR)

X_train, y_train = split_xy(train, "label")
X_test, y_test = split_xy(test, "label")
X_train, X_test = align_columns(X_train, X_test)

X_train = drop_identifier_like_columns(X_train)
X_test = drop_identifier_like_columns(X_test)

print("Train:", X_train.shape, " Test:", X_test.shape)

Train: (175341, 42)  Test: (82332, 42)


## Step 1 — Verify the label columns are gone from X

The first and most embarrassing form of leakage is training on the answer. `split_xy` drops both `label` and `attack_cat`, but never assume — assert it.

`attack_cat` matters as much as `label` here: it encodes the answer to the binary question too. If `attack_cat` is anything other than "Normal", the label is 1.

Watch the correlation table below: `sttl` tops it. Lab 00b showed why: TTL largely records which testbed machine generated the traffic. A strong correlation is not leakage by itself, but it is a shortcut worth naming.

In [3]:
forbidden = ["label", "attack_cat"]

for name, frame in [("X_train", X_train), ("X_test", X_test)]:
    leaked = [c for c in forbidden if c in frame.columns]
    print(f"{name}: leaked target columns -> {leaked if leaked else 'none'}")
    assert not leaked, f"{name} still contains a target column"

# A feature perfectly correlated with the target is also suspicious.
numeric = X_train.select_dtypes(include="number")
correlations = numeric.corrwith(y_train).abs().sort_values(ascending=False)

print("\nMost target-correlated features (|r| > 0.9 deserves investigation):")
print(correlations.head(8).round(3).to_string())

X_train: leaked target columns -> none
X_test: leaked target columns -> none

Most target-correlated features (|r| > 0.9 deserves investigation):
sttl                0.693
ct_state_ttl        0.578
dload               0.394
ct_dst_sport_ltm    0.357
dmean               0.342
rate                0.338
swin                0.334
dwin                0.320


## Step 2 — Fit preprocessing on the training data only

This is the pattern from the lesson. The important line is the `fit` on `X_train` — the preprocessor learns categories and scaling parameters **only** from training data, then applies those learned rules to both sets.

In [4]:
preprocessor = build_preprocessor(X_train, scale_numeric=False)
model = RandomForestClassifier(
    n_estimators=250, class_weight="balanced_subsample",
    random_state=42, n_jobs=-1,
)

preprocessor.fit(X_train)

X_train_ready = preprocessor.transform(X_train)
X_test_ready  = preprocessor.transform(X_test)

model.fit(X_train_ready, y_train)
y_pred = model.predict(X_test_ready)

honest = binary_metrics(y_test, y_pred)
display(binary_metrics_frame("train-only preprocessing", honest).round(4))

,accuracy,precision,recall,f1,false_positive_rate,false_negative_rate,roc_auc
model,,,,,,,
train-only preprocessing,0.8708,0.8163,0.9876,0.8938,0.2722,0.0124,None


## Step 3 — The dangerous shortcut, and why it barely shows here

Now do it wrong on purpose. Fit the preprocessor on train and test combined, the move that feels harmless because "it's only preprocessing, not the model."

Expect a tiny gap, and know **why**: this model is a random forest with no scaling, and the data has no missing values. Trees ignore scaling and there is nothing to impute, so this particular mistake has almost nothing to leak *here*. With a linear model, a scaler, or imputation, the same mistake leaks test statistics into training. The habit is wrong even when the number does not move.

In [5]:
leaky_preprocessor = build_preprocessor(X_train, scale_numeric=False)

combined = pd.concat([X_train, X_test], axis=0)
leaky_preprocessor.fit(combined)          # <-- the mistake: test data shaped the transform

leaky_model = RandomForestClassifier(
    n_estimators=250, class_weight="balanced_subsample",
    random_state=42, n_jobs=-1,
).fit(leaky_preprocessor.transform(X_train), y_train)

leaky_pred = leaky_model.predict(leaky_preprocessor.transform(X_test))
leaked = binary_metrics(y_test, leaky_pred)

display(pd.concat([
    binary_metrics_frame("train-only (honest)", honest),
    binary_metrics_frame("fit on train+test (leaky)", leaked),
]).round(4))

print(f"F1 difference: {leaked.f1 - honest.f1:+.4f}")
print("Small here because trees ignore scaling and nothing needed imputing. The habit is still wrong: the next demo shows a leak that does move the score.")

,accuracy,precision,recall,f1,false_positive_rate,false_negative_rate,roc_auc
model,,,,,,,
train-only (honest),0.8708,0.8163,0.9876,0.8938,0.2722,0.0124,None
fit on train+test (leaky),0.8709,0.8166,0.9872,0.8939,0.2716,0.0128,None


F1 difference: +0.0001
Small here because trees ignore scaling and nothing needed imputing. The habit is still wrong: the next demo shows a leak that does move the score.


### A leak that does move the number: random re-splitting

The official UNSW-NB15 partition was published as a fixed train/test split. A common shortcut is to pool everything and re-split at random. With this many duplicate rows, copies of the same flow land on both sides, and the model gets graded on rows it memorized.

In [6]:
from sklearn.model_selection import train_test_split

X_pool = pd.concat([X_train, X_test], ignore_index=True)
y_pool = pd.concat([y_train, y_test], ignore_index=True)
X_tr_r, X_te_r, y_tr_r, y_te_r = train_test_split(
    X_pool, y_pool, test_size=0.30, stratify=y_pool, random_state=42,
)

resplit = Pipeline([
    ("prep", build_preprocessor(X_tr_r, scale_numeric=False)),
    ("model", RandomForestClassifier(
        n_estimators=250, class_weight="balanced_subsample",
        random_state=42, n_jobs=-1)),
]).fit(X_tr_r, y_tr_r)
resplit_metrics = binary_metrics(y_te_r, resplit.predict(X_te_r))

tr_keys = set(pd.util.hash_pandas_object(X_tr_r, index=False))
te_keys = pd.util.hash_pandas_object(X_te_r, index=False)
print(f"Random-split test rows also present in its training data: {te_keys.isin(tr_keys).mean():.1%}")

display(pd.concat([
    binary_metrics_frame("official split (honest)", honest),
    binary_metrics_frame("pooled random re-split (leaky)", resplit_metrics),
]).round(4))
print(f"F1 difference: {resplit_metrics.f1 - honest.f1:+.4f}")

Random-split test rows also present in its training data: 44.6%


,accuracy,precision,recall,f1,false_positive_rate,false_negative_rate,roc_auc
model,,,,,,,
official split (honest),0.8708,0.8163,0.9876,0.8938,0.2722,0.0124,None
pooled random re-split (leaky),0.9505,0.9624,0.9599,0.9612,0.0663,0.0401,None


F1 difference: +0.0674


## Step 4 — Use a scikit-learn Pipeline

The Pipeline links preprocessing and the classifier into one ordered object. Call `fit` and preprocessing is learned from the training data only, then flows into the model. Call `predict` and the same learned transform is applied to new data.

You cannot accidentally fit on the test set, because you never touch the preprocessor yourself.

In [7]:
classifier = RandomForestClassifier(
    n_estimators=250, class_weight="balanced_subsample",
    random_state=42, n_jobs=-1,
)

from sklearn.pipeline import Pipeline

pipe = Pipeline([
    ('prep', preprocessor),
    ('model', classifier)
])

pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)

pipeline_metrics = binary_metrics(y_test, y_pred)
display(binary_metrics_frame("Pipeline", pipeline_metrics).round(4))

print("Matches the manual train-only result:",
      np.isclose(pipeline_metrics.f1, honest.f1, atol=1e-9))
print("Same discipline, fewer ways to get it wrong.")

,accuracy,precision,recall,f1,false_positive_rate,false_negative_rate,roc_auc
model,,,,,,,
Pipeline,0.8708,0.8163,0.9876,0.8938,0.2722,0.0124,None


Matches the manual train-only result: True
Same discipline, fewer ways to get it wrong.


## Step 5 — Check for duplicate rows across the split

If a near-identical row appears in both train and test, the model has effectively seen the answer. Test performance goes up for no honest reason.

In [8]:
train_hash = pd.util.hash_pandas_object(X_train, index=False)
test_hash  = pd.util.hash_pandas_object(X_test, index=False)

overlap = set(train_hash).intersection(set(test_hash))
print('Duplicate overlap:', len(overlap))

# How many test ROWS does that overlap actually account for?
overlapping_rows = test_hash.isin(overlap)
n_overlap = int(overlapping_rows.sum())

print(f"Duplicate rows inside train : {train_hash.duplicated().sum()}")
print(f"Duplicate rows inside test  : {test_hash.duplicated().sum()}")
print(f"Test rows also seen in train: {n_overlap}  ({n_overlap / len(X_test):.2%} of the test set)")

Duplicate overlap: 1302


Duplicate rows inside train : 74301
Duplicate rows inside test  : 28386
Test rows also seen in train: 8541  (10.37% of the test set)


## Step 6 — Related entities crossing the split

Exact duplicates are the easy case. The harder one is *related* rows: two records belonging to the same underlying entity, one in train and one in test. The model memorizes the entity instead of learning the behaviour.

UNSW-NB15's prepared partition ships no session or host identifier, so we cannot group by entity here — and saying that out loud is part of the audit. The check below approximates it with a signature built only from categorical and count fields, ignoring the continuous and random ones, and looks for test rows whose signature also appears in training.

In [9]:
# Build the signature only from categorical and count fields. Continuous and
# random fields (TCP base sequence numbers, loads, jitter, inter-packet times)
# make almost every row unique and would hide related rows.
stable_cols = [c for c in X_train.columns
               if c in ["proto", "service", "state", "sttl", "dttl", "swin", "dwin",
                        "trans_depth", "is_ftp_login", "ct_ftp_cmd", "ct_flw_http_mthd",
                        "is_sm_ips_ports"]
               or c.startswith("ct_")]

train_stable = pd.util.hash_pandas_object(X_train[stable_cols], index=False)
test_stable = pd.util.hash_pandas_object(X_test[stable_cols], index=False)

near_overlap = set(train_stable)
n_near = int(test_stable.isin(near_overlap).sum())

print("Signature fields:", ", ".join(stable_cols))
print(f"Test rows sharing a stable-field signature with train: {n_near} "
      f"({n_near / len(X_test):.2%})  vs exact duplicates: {n_overlap} ({n_overlap / len(X_test):.2%})")
print()
print("This is a proxy, not proof. Record the limitation explicitly:")
print("no session/host identifier exists, so entity-level leakage cannot be ruled out.")
print("Also ask: stcpb and dtcpb are random TCP sequence numbers. Should they be model inputs at all?")

Signature fields: proto, service, state, sttl, dttl, swin, dwin, trans_depth, ct_srv_src, ct_state_ttl, ct_dst_ltm, ct_src_dport_ltm, ct_dst_sport_ltm, ct_dst_src_ltm, is_ftp_login, ct_ftp_cmd, ct_flw_http_mthd, ct_src_ltm, ct_srv_dst, is_sm_ips_ports
Test rows sharing a stable-field signature with train: 44533 (54.09%)  vs exact duplicates: 8541 (10.37%)

This is a proxy, not proof. Record the limitation explicitly:
no session/host identifier exists, so entity-level leakage cannot be ruled out.
Also ask: stcpb and dtcpb are random TCP sequence numbers. Should they be model inputs at all?


## Step 7 — Correct what you found, then re-score

If a check found something, fix it and recalculate. A number that moves after a correction is evidence. A number you never re-ran is just a claim.

In [10]:
if n_overlap:
    clean = ~overlapping_rows.values
    corrected = binary_metrics(
        y_test[clean],
        pipe.predict(X_test[clean]),
        pipe.predict_proba(X_test[clean])[:, 1],
    )
    display(pd.concat([
        binary_metrics_frame("as reported (full test set)", pipeline_metrics),
        binary_metrics_frame("corrected (duplicates removed)", corrected),
    ]).round(4))
    print(f"F1 moved {corrected.f1 - pipeline_metrics.f1:+.4f} once duplicates were removed.")
    reported = corrected
else:
    print("No exact overlap to correct — report the check as performed and passed.")
    print("A negative result is still evidence, provided you actually ran it.")
    reported = pipeline_metrics

,accuracy,precision,recall,f1,false_positive_rate,false_negative_rate,roc_auc
model,,,,,,,
as reported (full test set),0.8708,0.8163,0.9876,0.8938,0.2722,0.0124,None
corrected (duplicates removed),0.8617,0.7961,0.9858,0.8808,0.2719,0.0142,0.978346


F1 moved -0.0130 once duplicates were removed.


## Step 8 — Try to break your own result

Shuffle the labels and retrain. If the data carries real signal and nothing else, ranking quality should **collapse to chance: ROC AUC close to 0.50.** Read the result in *both* directions. Far above 0.5 means something still predicts the answer without labels. Far *below* 0.5 means the label-free model still ranks the test set systematically. Either way, something is leaking structure.

This is the cheapest credibility check in the whole course, and the one reviewers respect most.

In [11]:
def shuffled_label_auc(X_fit, y_fit, n_estimators=100):
    rng = np.random.default_rng(42)
    y_shuffled = rng.permutation(np.asarray(y_fit))
    model = Pipeline([
        ("prep", build_preprocessor(X_fit, scale_numeric=False)),
        ("model", RandomForestClassifier(
            n_estimators=n_estimators, class_weight="balanced_subsample",
            random_state=42, n_jobs=-1)),
    ]).fit(X_fit, y_shuffled)
    score = model.predict_proba(X_test)[:, 1]
    return binary_metrics(y_test, (score >= 0.5).astype(int), score)

shuffled_metrics = shuffled_label_auc(X_train, y_train)
real_score = pipe.predict_proba(X_test)[:, 1]
real_metrics = binary_metrics(y_test, y_pred, real_score)

display(pd.concat([
    binary_metrics_frame("real labels", real_metrics),
    binary_metrics_frame("shuffled labels", shuffled_metrics),
]).round(4))

# Judge this on ROC AUC, not F1: a useless model can post a respectable F1 by
# favouring the majority class. Destroy the labels and AUC must land near 0.50.
print(f"Attack base rate in test : {y_test.mean():.3f}")
print(f"Real-label AUC           : {real_metrics.roc_auc:.3f}")
print(f"Shuffled-label AUC       : {shuffled_metrics.roc_auc:.3f}  (0.50 = chance)")

distance = abs(shuffled_metrics.roc_auc - 0.5)
print("\nPASS: shuffled labels collapse to chance." if distance < 0.05 else
      f"\nINVESTIGATE: a label-free model is {distance:.2f} away from chance. "
      "Suspect duplicate rows or an ID-like feature.")

,accuracy,precision,recall,f1,false_positive_rate,false_negative_rate,roc_auc
model,,,,,,,
real labels,0.8708,0.8163,0.9876,0.8938,0.2722,0.0124,0.9805
shuffled labels,0.4677,0.5105,0.8094,0.6261,0.9509,0.1906,0.3238


Attack base rate in test : 0.551
Real-label AUC           : 0.981
Shuffled-label AUC       : 0.324  (0.50 = chance)

INVESTIGATE: a label-free model is 0.18 away from chance. Suspect duplicate rows or an ID-like feature.


### Step 8b — Find the cause, and prove it

A result far *below* chance has a classic cause in this dataset: duplicates. Rows repeated many times in training receive many random labels, which average out to middling scores, while unique rows get extreme memorized scores. Test rows that match training duplicates are mostly attacks, so they end up ranked *lower*, and AUC falls under 0.5.

The test of that explanation: remove duplicate rows from the training data and run the same control again.

In [12]:
dedup_mask = ~X_train.duplicated()
dedup_shuffled = shuffled_label_auc(X_train[dedup_mask], y_train[dedup_mask])

seen = test_hash.isin(set(train_hash))
print(f"Attack share of test rows also seen in train : {y_test[seen.values].mean():.3f}")
print(f"Attack share of test rows never seen in train: {y_test[~seen.values].mean():.3f}")
print()
print(f"Shuffled-label AUC, full training data  : {shuffled_metrics.roc_auc:.3f}")
print(f"Shuffled-label AUC, duplicates removed : {dedup_shuffled.roc_auc:.3f}  (0.50 = chance)")
print("\nThe sanity check flagged the duplicate problem before the duplicate check measured it.")

Attack share of test rows also seen in train : 0.828
Attack share of test rows never seen in train: 0.519

Shuffled-label AUC, full training data  : 0.324
Shuffled-label AUC, duplicates removed : 0.474  (0.50 = chance)

The sanity check flagged the duplicate problem before the duplicate check measured it.


## Step 9 — Write the methods paragraph

From the lesson:

- **Bad:** "Model achieved 99% accuracy."
- **Better:** "Model was evaluated on held-out test data with train-only preprocessing."
- **Best:** "Error analysis and leakage checks were documented."

Fill this in with your own numbers and save it. This paragraph is what makes every earlier lab believable.

In [13]:
methods = f"""# Lab 07 — Methods & Leakage Audit

## Methods paragraph (this is the part that goes in the write-up)
The model was trained on the official UNSW-NB15 prepared training partition
({len(X_train)} rows) and evaluated on the held-out testing partition
({len(X_test)} rows); no random re-splitting was performed. Both target columns
(`label`, `attack_cat`) were removed from the feature matrix and this was
asserted programmatically. All preprocessing (median imputation and one-hot
encoding; no scaling, since the model is tree-based) was fitted on training data only and applied to the test set through
a scikit-learn Pipeline, so no test-set statistics influenced training. Exact
duplicate rows shared across the split were counted ({n_overlap},
{n_overlap / len(X_test):.2%} of the test set) and metrics were recalculated
with them removed. A shuffled-label control was run as a sanity check: ROC AUC
fell from {real_metrics.roc_auc:.3f} to {shuffled_metrics.roc_auc:.3f}, well below chance (0.50),
which indicates leaked structure. Repeating the control with duplicate
training rows removed gave {dedup_shuffled.roc_auc:.3f}, identifying duplicates as the cause.

## Leakage checks performed
- target columns removed from X ......... asserted
- preprocessing fitted on train only .... via Pipeline
- exact duplicate overlap ............... {n_overlap} test rows
- approximate entity overlap ............ {n_near} test rows (proxy only)
- shuffled-label control ................ AUC {shuffled_metrics.roc_auc:.3f} (dedup: {dedup_shuffled.roc_auc:.3f})
- pooled random re-split ................ F1 {resplit_metrics.f1:.4f} vs official {honest.f1:.4f}

## Leakage risks I could NOT rule out
1. No session or host identifier exists, so entity-level leakage across the
   split cannot be excluded.
2.

## What this validation cannot prove
The dataset carries no timestamps, so train-on-past / test-on-future was not
possible. These results therefore say nothing about how the model would hold up
against traffic that drifts over time.

## Reported headline metric (after correction)
F1 {reported.f1:.4f} | recall {reported.recall:.4f} | FNR {reported.false_negative_rate:.4f}
"""

methods_path = RESULTS_DIR / "lab07_methods_and_leakage_audit.md"
methods_path.write_text(methods)
print("Saved:", methods_path)
print(methods)

Saved: /home/claude/run/l07/results/lab07_methods_and_leakage_audit.md
# Lab 07 — Methods & Leakage Audit

## Methods paragraph (this is the part that goes in the write-up)
The model was trained on the official UNSW-NB15 prepared training partition
(175341 rows) and evaluated on the held-out testing partition
(82332 rows); no random re-splitting was performed. Both target columns
(`label`, `attack_cat`) were removed from the feature matrix and this was
asserted programmatically. All preprocessing (median imputation and one-hot
encoding; no scaling, since the model is tree-based) was fitted on training data only and applied to the test set through
a scikit-learn Pipeline, so no test-set statistics influenced training. Exact
duplicate rows shared across the split were counted (8541,
10.37% of the test set) and metrics were recalculated
with them removed. A shuffled-label control was run as a sanity check: ROC AUC
fell from 0.981 to 0.324, well below chance (0.50),
which indicates leaked 

## Your analysis

1. Why did leaky preprocessing barely move the score with this model, while random re-splitting moved it more? Would a linear model with scaling behave the same way?
2. What did the duplicate check find, and how would you report a result of zero?
3. The shuffled-label AUC landed far below 0.5. Explain in your own words why duplicates push it *below* chance, not above.
4. This dataset ships with no timestamps, so train-on-past / test-on-future is not available here. What would you need to add, and what would that change about your claims?
5. Rewrite one sentence from an earlier lab's conclusion so that it survives a leakage-aware reader.

**Deliverables:** target-column assertion, train-only preprocessing, a `Pipeline`, a duplicate-overlap check, a shuffled-label control, and written validation notes with the leakage risks you could not rule out.

## From lab to SOC

Leakage is how detection projects fail after launch: the model looked excellent in testing, then performance drops on live traffic and nobody can explain why. A documented leakage audit (Step 9) is what lets a reviewer, a manager, or an auditor trust your result.

**Check yourself**
1. Why does a random re-split of this dataset inflate the score so much?
2. A colleague shows a detector with 99.9% recall on "held-out" data. What three questions do you ask first?
3. What is the security-testing equivalent of a shuffled-label control?